# Crucial `stringjax_tools` Features

This notebook is the shortest executable tour of the package.  The point of
`stringjax_tools` is not to encode any particular physics or geometry
convention.  Instead, it provides reusable JAX mechanics that downstream
packages can wrap with their own shape names, pytree state policy, and cache
settings.

The examples below are deliberately small.  They show the standard pattern:
write a clean single-sample function, declare how selected inputs may be
batched, keep application-specific names in the consuming package, and make
cache/JIT/pytree decisions explicit.


In [ ]:
import tempfile
from pathlib import Path

import jax
import jax.numpy as jnp

from stringjax_tools import (
    PytreePolicy,
    auto_vmap,
    configure_compilation_cache,
    jit_with_static_args,
    vmapping_func_cached,
)


## 1. Rank-Checked Automatic Vectorisation

`auto_vmap` decorates a function that is naturally written for one sample.  A
sample rank of `1` means that a vector such as `(dim,)` is one input and
`(N, dim)` is a leading-axis batch.  A sample rank of `0` means that a scalar
`()`, or equivalently a zero-dimensional JAX array, is one input and `(N,)` is a
leading-axis batch.

If several checked arguments are batched, their leading batch sizes must agree.
Unbatched checked arguments are broadcast with `in_axes=None`.


In [ ]:
@auto_vmap(moduli=1, tau=0, fluxes=1)
def observable(moduli, tau, fluxes, scale=1.0):
    return scale * (
        jnp.sum(jnp.abs(moduli) ** 2)
        + jnp.real(tau)
        + 0.01 * jnp.sum(fluxes**2)
    )

moduli = jnp.array([1.0, 2.0])
tau = jnp.array(3.0)
fluxes = jnp.arange(6.0)

batched_moduli = jnp.stack([moduli, moduli + 1.0])
batched_tau = jnp.array([tau, tau + 1.0])
batched_fluxes = jnp.stack([fluxes, fluxes + 1.0])

print('single:', observable(moduli, tau, fluxes))
print('paired batch:', observable(batched_moduli, batched_tau, batched_fluxes))
print('batched moduli, broadcast tau and fluxes:', observable(batched_moduli, tau, fluxes))


## 2. Keep Package Conventions Local

A downstream package usually has recurring names such as `moduli`, `tau`, or
`fluxes`, but the expected lengths are package-specific.  The recommended
pattern is to define a tiny package-local wrapper that selects names and
trailing-shape rules, while `stringjax_tools` remains generic.

Here `dim` and `flux_dim` are just attributes of the local toy model.  In a real
package they would be whatever dimensions that package already owns.


In [ ]:
PACKAGE_RANKS = {
    'moduli': 1,
    'tau': 0,
    'fluxes': 1,
}

PACKAGE_SHAPES = {
    'moduli': 'dim',
    'fluxes': 'flux_dim',
}


def package_auto_vmap(*names, jit=True):
    sample_ranks = {name: PACKAGE_RANKS[name] for name in names}
    sample_shapes = {
        name: PACKAGE_SHAPES[name]
        for name in sample_ranks
        if name in PACKAGE_SHAPES
    }
    return auto_vmap(
        sample_ranks=sample_ranks,
        sample_shapes=sample_shapes,
        validate_shapes=True,
        jit=jit,
    )


class ToyModel:
    def __init__(self, dim, flux_dim):
        self.dim = dim
        self.flux_dim = flux_dim

    @package_auto_vmap('moduli', 'fluxes')
    def score(self, moduli, fluxes):
        return jnp.sum(moduli) + 0.1 * jnp.sum(fluxes)


model = ToyModel(dim=2, flux_dim=6)
print('validated batch:', model.score(jnp.ones((3, 2)), jnp.ones((3, 6))))

try:
    model.score(jnp.ones(2), jnp.ones(5))
except ValueError as exc:
    print('shape error:', str(exc).splitlines()[0])


Derived dimensions should be ordinary Python callables rather than expression
strings.  A callable receives the bound argument mapping, so it can inspect
`self` or any other already-bound argument.


In [ ]:
class DerivedFluxModel:
    def __init__(self, n_fluxes):
        self.n_fluxes = n_fluxes

    @auto_vmap(
        sample_ranks={'fluxes': 1},
        sample_shapes={'fluxes': lambda bound: (2 * bound['self'].n_fluxes,)},
        validate_shapes=True,
    )
    def norm(self, fluxes):
        return jnp.linalg.norm(fluxes)


derived_model = DerivedFluxModel(n_fluxes=3)
print('derived shape:', derived_model.norm(jnp.ones((4, 6))))


## 3. Explicit Cached Manual `vmap`

Use `vmapping_func_cached` when the batching axes are already part of the design.
This is the manual counterpart to `auto_vmap`: it does not infer axes; it caches
a `jax.jit(jax.vmap(...))` wrapper for a fixed `in_axes` pattern and static
keyword configuration.


In [ ]:
def affine_square(x, offset=0.0):
    return x * x + offset

batched_affine = vmapping_func_cached(affine_square, in_axes=0, offset=2.0)
print('manual vmap:', batched_affine(jnp.arange(4.0)))
print('cached object reused:', batched_affine is vmapping_func_cached(affine_square, in_axes=0, offset=2.0))


## 4. Static-Argument JIT Helpers

Use `jit_with_static_args` when the static positions are known.  This is usually
preferable to dynamic static-argument detection in library code, because the
static boundary is explicit and easier to reason about.


In [ ]:
def truncated_power_sum(x, degree, shift):
    result = jnp.zeros_like(x) + shift
    for power in range(1, degree + 1):
        result = result + x**power
    return result

truncated_power_sum_jit = jit_with_static_args(truncated_power_sum, static_argnums=(1,))
print('static degree:', truncated_power_sum_jit(jnp.array([1.0, 2.0]), 3, 0.5))


## 5. Pytree Policies for Stateful Classes

JAX reconstructs custom pytree objects without calling `__init__`.  A
`PytreePolicy` makes the state split explicit:

- static configuration is stored in JAX auxiliary data;
- array-like numerical state is traced as pytree children;
- ignored eager caches are restored with safe defaults when needed.

Do not ignore semantic state.  Ignore only recomputable caches or scratch state.


In [ ]:
class StatefulModel:
    def __init__(self, dim, weights):
        self.dim = dim
        self.weights = jnp.asarray(weights)
        self._cache = {'built_eagerly': True}


policy = PytreePolicy(static_keys=('dim',), ignore_defaults={'_cache': dict})
policy.register(StatefulModel)

stateful = StatefulModel(dim=2, weights=jnp.array([1.0, 2.0]))
leaves, treedef = jax.tree_util.tree_flatten(stateful)
restored = jax.tree_util.tree_unflatten(treedef, leaves)

print('dynamic leaves:', leaves)
print('restored dim:', restored.dim)
print('restored weights:', restored.weights)
print('restored cache:', restored._cache)


## 6. Persistent Compilation Cache Setup

Compilation-cache setup has no import-time side effects.  Call
`configure_compilation_cache` near the top of a real script or notebook, before
any JAX compilation happens.  This tutorial has already compiled functions, so
the setup cell is disabled by default.


In [ ]:
RUN_CACHE_SETUP = False

if RUN_CACHE_SETUP:
    cache_dir = Path(tempfile.mkdtemp(prefix='stringjax_tools_cache_'))
    applied = configure_compilation_cache(
        cache_dir=cache_dir,
        max_size_bytes=1_000_000_000,
    )
    print('applied cache settings:', applied)
else:
    print('Set RUN_CACHE_SETUP=True before the first JAX compilation in a real workflow.')


## Take-Away

Use `auto_vmap` for user-facing scalar-or-batched calls, keep package-specific
shape conventions in the consuming package, keep deliberate manual `vmap`
kernels explicit, mark JIT-static arguments intentionally, and register
stateful model classes with a documented pytree policy.
